# Step 25 — GSE135779 as pseudobulk: one column per child

**Data type: scRNA_seq** (GSE135779), used as pseudobulk. **Reads:** `data/GSE135779/`.
**Writes:** `step25_GSE135779.rds`.

A single-cell matrix has one column per cell. **Pseudobulk** sums the counts of all cells from one
donor, which gives one column per child, like a bulk sample of that child's PBMC. Summing over all cells
needs no cell-type labels, which GEO does not provide.

The pseudobulk counts are then normalised exactly as the bulk counts in step 23: filterByExpr, TMM,
voom log2 counts per million. Only children are kept: 33 with SLE and 11 healthy.

**PBMC is not whole blood.** Density-gradient separation removes most mature neutrophils and red
cells. Low-density granulocytes, which are raised in lupus, do sediment with PBMC. Step 27 tests
whether the granulocyte endotype (E2) is visible here at all.

In [1]:
source("../src/paths.R")
source("../src/platforms.R")
suppressMessages({library(GEOquery); library(Biobase)})
pd <- pData(suppressMessages(getGEO(filename = raw("GSE135779", "GSE135779_series_matrix.txt.gz"), getGPL = FALSE)))
meta <- data.frame(sample = rownames(pd), donor = sub(" \\[.*", "", pd$title),
                   age = as.numeric(pd[["age:ch1"]]), age_group = pd[["age group:ch1"]],
                   disease = factor(ifelse(pd[["groups:ch1"]] == "SLE", "SLE", "Healthy"), levels = c("Healthy", "SLE")),
                   row.names = rownames(pd))
table(meta$age_group, meta$disease)
meta <- meta[meta$age_group == "Children", ]
summary(meta$age)

          
           Healthy SLE
  Adult          5   7
  Children      11  33

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   7.00   13.00   16.00   14.95   17.00   19.00 

## Sum counts per child

About a minute: 44 sparse matrices, each read once and summed across its cells.

In [2]:
genes <- read.delim(raw("GSE135779", "GSE135779_genes.tsv.gz"), header = FALSE, col.names = c("ensembl", "symbol"))
files <- list.files(raw("GSE135779", "raw"), pattern = "matrix.mtx.gz$", full.names = TRUE)
files <- setNames(files, sub("_.*", "", basename(files)))[rownames(meta)]
stopifnot(!anyNA(files))
cells <- c()
counts <- sapply(names(files), function(g) {
  m <- Matrix::readMM(gzfile(files[[g]])); stopifnot(nrow(m) == nrow(genes))
  cells[g] <<- ncol(m)
  Matrix::rowSums(m)
})
rownames(counts) <- genes$ensembl
meta$cells <- cells[rownames(meta)]
summary(meta$cells)

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   2965    5088    5904    6451    7224   13834 

## Ensembl identifiers to the shared gene symbols

Ensembl → Entrez → current NCBI symbol, through NCBI gene_info, as for the other two studies. Genes
without a mapping keep the symbol shipped with the study.

In [3]:
gi  <- read_gene_info()
sym <- gi$Symbol[match(genes$ensembl, gi$ensembl)]
sym[is.na(sym)] <- genes$symbol[is.na(sym)]
counts <- collapse_max_mean(counts, sym)
norm <- counts_to_logcpm(counts, group = meta$disease)
E <- norm$E
c(genes_in = norm$genes_in, genes_measured = norm$genes_kept, children = ncol(E))
read_gene_set("ifn-type1-6.txt") %in% rownames(E)

genes_in genes_measured       children 
         32614          14409             44

[1] TRUE TRUE TRUE TRUE TRUE TRUE

In [4]:
saveRDS(list(E = E, meta = meta, data_type = "scRNA_seq pseudobulk", tissue = "PBMC"), art("step25_GSE135779.rds"))
cat("wrote", art("step25_GSE135779.rds"), "\n")

wrote /Users/adeslatt/Scitechcon Dropbox/Anne DeslattesMays/projects/endotypes-transcriptomics/data/run_artifacts/step25_GSE135779.rds 


## Findings

Written after the run.